# Análisis de consumo para evadir contraseña (PRINCIPAL)

**RESUMEN:** *Este tutorial te introducirá a vulnerar dispositivos determinando cuándo un dispositivo está realizando ciertas operaciones. Nuestro dispositivo objetivo realizará una comprobación simple de contraseña, y demostraremos cómo llevar a cabo un análisis de potencia básico.*

**OBJETIVOS DE APRENDIZAJE:**

* Cómo se puede utilizar la potencia para determinar información temporal (*timing*).
* Graficar múltiples iteraciones mientras se varían los datos de entrada para localizar puntos de interés.
* Utilizar la diferencia de formas de onda para encontrar puntos de interés.
* Realizar capturas de potencia con el hardware ChipWhisperer.

## Recopilación de trazas de potencia

Para la inicialización del hardware (tanto el osciloscopio como el target) como para definir la función que nos ayudará a enviar el password al target así como capturar las trazas de potencia, nos ayudamos del notebook con el mismo título que este pero terminado en `(HARDWARE)`.

Antes de ejecutarlo, échale un vistazo para entender qué es lo que hace !!!

Además, échale también un vistazo a la función `main()` del firmware `basic-passwdcheck.c` que es el que se está cargando en el target, para ver cómo chequea los passwords enviados para determinar si son o no correctos.

Una vez ejecutado, asegúrate de que no ha habido ningún problema y el firmware se ha escrito en el target (`Verified flash OK...`)

In [ ]:
%run 3.Analisis_de_consumo_para_evadir_password_(HARDWARE).ipynb

El siguiente código hace una prueba de captura de traza de potencia enviando el password `h` y chequeando que el tamaño de la traza capturada sea `3000`. 

¡Asegúrate de obtener el mensaje `"✔️ OK to continue!"` una vez que ejecutes la siguiente celda; de lo contrario, fallará más adelante!

In [ ]:
trace_test = cap_pass_trace("h\n")

#Basic sanity check
assert(len(trace_test) == 3000)
print("✔️ OK to continue!")

## Exploración

Entonces, ¿qué podemos hacer con esto? Bueno, en primer lugar, vamos a hacer trampa ya que vamos a saber que tenemos una contraseña preestablecida que empieza por `h` y tiene 5 caracteres de longitud. Pero esa va a ser la única pista por ahora; ¿qué puedes hacer? Pues de entrada, vamos a intentar graficar una comparación de `h` con cualquier otra cosa.

Si necesitas recordar cómo hacer un gráfico (`cw.plot`), consulta el notebook `2.Diferencias_consumo_potencia_(PRINCIPAL)`.

La siguiente celda te muestra cómo capturar una traza de potencia enviando `h` como contraseña. A partir de ahí:

1. Intenta añadir el código para graficar y mira qué aspecto tiene.
2. Envía diferentes contraseñas al dispositivo. Por ahora, solo nos centraremos en la diferencia entre una contraseña que empiece por `h` y otra diferente.
3. Grafica las distintas formas de onda (muestra la forma de onda cuando se ha enviado `h` contra la de otro carácter).

In [ ]:
#Example - capture 'h' - end with newline '\n' as serial protocol expects that
trace_h = cap_pass_trace("h\n")

print(trace_h)

# ###################
# Add your code here
# ###################
raise NotImplementedError("Add your code here, and delete this.")

A modo de referencia, la salida debería verse aproximadamente así:

<img src="img/spa_password_h_vs_0_overview.png" alt="SPA of Power Analysis" width="450"/>

Si estás utilizando la función mágica `%matplotlib notebook`, puedes hacer zoom al inicio. Lo que debes notar es que se toman dos rutas de ejecución de código diferentes, según si el camino es correcto o incorrecto. Aquí, por ejemplo, se muestra el procesamiento de un carácter correcto frente a uno incorrecto:

<img src="img/spa_password_h_vs_0_zoomed.png" alt="SPA of Power Analysis" width="450"/>

Muy bien, interesante... ¿qué sigue? Grafiquemos cada posible carácter de contraseña que podríamos enviar.

Nuestra implementación de contraseña solo reconoce los caracteres de la lista `abcdefghijklmnopqrstuvwxyz0123456789`, por lo que nos limitaremos a esos caracteres válidos por ahora.

Escribe algo de código en el siguiente bloque que implemente este algoritmo:

    for CHARACTER in LIST_OF_VALID_CHARACTERS:
        trace = cap_pass_trace(CHARACTER + "\n")
        plot(trace)
        
Lo anterior no es código del todo válido, ¡así que ajústalo para que funcione! También notarás que las trazas son demasiado largas; tal vez prefieras hacer un gráfico más acotado que solo incluya, por ejemplo, las primeras 500 (`[0:500]`) muestras de la traza de potencia.

In [ ]:
# ###################
# Add your code here
# ###################
raise NotImplementedError("Add your code here, and delete this.")

El resultado final debería ser que, si haces zoom, verás una zona donde una única traza discordante (*outlier*) no sigue la trayectoria del resto de las trazas. Esa es una excelente noticia, ya que significa que podemos extraer información sobre el sistema a partir del análisis de potencia.

<img src="img/spa_password_list_char1.png" alt="SPA of Power Analysis against all inputs" width="450"/>

Usando tu bucle, también puedes intentar modificar el análisis para capturar un "primer" carácter correcto y, a continuación, todos los demás segundos caracteres incorrectos. ¿Aprecias alguna diferencia que puedas detectar?

El pseudocódigo se vería aproximadamente así:

    for CHARACTER in LIST_OF_VALID_CHARACTERS:
        trace = cap_pass_trace("h" + CHARACTER + "\n")
        plot(trace)

Pruébalo en tu bloque de código anterior y, después, intentemos automatizar este ataque para comprender los datos un poco mejor.

## Automatización de un ataque contra un único carácter

Para empezar, vamos a automatizar un ataque contra un **único** carácter de la contraseña. Dado que no conocemos la contraseña (supongamos), utilizaremos la estrategia de comparar todas las posibles entradas entre sí.

Una forma sencilla de hacer esto podría ser utilizar algo que sepamos que no puede formar parte de la contraseña válida. Siempre que se procese de la misma manera, esto funcionará perfectamente. Así que, por ahora, usemos como contraseña `0x00` (es decir, un byte nulo). Podemos comparar el procesamiento del byte nulo frente al de cualquier otro valor:

In [ ]:
ref_trace = cap_pass_trace("\x00\n")[0:1000]
fig = cw.plot(ref_trace)
other_trace = cap_pass_trace("c\n")[0:1000]
fig *= cw.plot(other_trace)
fig

Esto graficará una traza con una entrada de "\x00": ¡una contraseña nula! Este es un carácter no válido y parece procesarse igual que cualquier otra contraseña incorrecta.

Hagamos esto un poco más evidente y grafiquemos la diferencia entre una referencia conocida y cada una de las demás capturas. Debes escribir algo de código que haga algo como esto:

```python
ref_trace = cap_pass_trace( "\x00\n")

for CHARACTER in LIST_OF_VALID_CHARACTERS:
    trace = cap_pass_trace(CHARACTER + "\n")
    plot(trace - ref_trace)
```

Nuevamente, es posible que necesites modificar esto un poco, como añadir código para crear una nueva `figure()`. Observa también en el ejemplo anterior cómo reduje el número de muestras.

In [ ]:
# ###################
# Add your code here
# ###################
raise NotImplementedError("Add your code here, and delete this.")

Perfecto, con suerte ahora verás una gran "diferencia". Debería verse aproximadamente así:
    
<img src="img/spa_password_diffexample.png" alt="SPA with Difference" width="450"/>

¿Qué hacemos ahora? Hagamos que esto detecte automáticamente una diferencia tan grande. Algunas funciones útiles para probar son `np.sum()` y `np.abs()`.

La primera obtendrá los valores absolutos:

```python
import numpy as np
np.abs([-1, -3, 1, -5, 6])

    Out[]: array([1, 3, 1, 5, 6])
```

La segunda sumará todos los números:

```python
import numpy as np    
np.sum([-1, -3, 1, -5, 6])

    Out[]: -2
```

Usar solo `np.sum()` significa que las diferencias positivas y negativas se cancelarán entre sí, por lo que es mejor hacer algo como `np.sum(np.abs(DIFF))` para obtener un valor adecuado que indique cuán "cercana" fue la coincidencia.

In [ ]:
import numpy as np
np.abs([-1, -3, 1, -5, 6])

In [ ]:
import numpy as np
np.sum([-1, -3, 1, -5, 6])

In [ ]:
np.sum(np.abs(<COPY DATA HERE>))

Tomando tu bucle anterior, modifícalo para imprimir un indicador (valor numérico) de cómo de cerca coincide esto con tu traza. Algo como lo siguiente debería funcionar:

    ref_trace = cap_pass_trace( "\x00\n")

    for CHARACTER in LIST_OF_VALID_CHARACTERS:
        trace = cap_pass_trace(CHARACTER + "\n")
        diff = SUM(ABS(trace - ref_trace))

        print("{:1} diff = {:2}".format(CHARACTER, diff))

In [ ]:
# ###################
# Add your code here
# ###################
raise NotImplementedError("Add your code here, and delete this.")

Ahora la parte fácil: modifica tu código anterior para imprimir automáticamente el carácter correcto de la contraseña. Esto debe hacerse comparando la variable `diff`; basándote en los caracteres impresos, deberías ver uno con un valor claramente superior a los demás. Establece un umbral razonable según los resultados de una ejecución.

## Ejecución de un ataque completo

Por último, rematemos la tarea. En lugar de atacar un único carácter, necesitamos atacar cada carácter de forma secuencial.

Si vuelves a la representación gráfica de las diferencias, puedes probar a utilizar el primer carácter correcto y el segundo carácter incorrecto. La idea básica es exactamente la misma que antes, pero ahora iteramos 5 veces y vamos construyendo la contraseña mediante fuerza bruta carácter a carácter.

Echa un vistazo al siguiente pseudocódigo básico:

    guessed_pw = "" #Store guessed password so far
    
    do a loop 5 times (max password size):
        
        ref_trace = capture power trace(guessed_pw + "\x00\n")
        
        for CHARACTER in LIST_OF_VALID_CHARACTERS:
            trace = capture power trace (guessed_pw + CHARACTER + newline)
            diff = SUM(ABS(trace - ref_trace))
            
            if diff > THRESHOLD:
                
                guessed_pwd += CHARACTER
                print(guessed_pw)
                
                break


In [ ]:
# ###################
# Add your code here
# ###################
raise NotImplementedError("Add your code here, and delete this.")

Deberías obtener una salida parecida a esta:

    h
    h0
    h0p
    h0px
    h0px3

Si es así... 🥳🥳🥳🥳🥳🥳🥳🥳🥳🥳🥳🥳🥳 ¡¡¡Enhorabuena, lo lograste!!!!

Si no es así, echa un vistazo a las sugerencias de resolución de problemas más abajo. 

---
## Resolución de problemas

Algunos problemas comunes con los que podrías encontrarte. En primer lugar, si obtienes una salida que continúa prediciendo el primer carácter:

### Siempre se obtiene 'h'

    h
    hh
    hhh
    hhhh
    hhhhh

Comprueba si, al ejecutar `cap_pass_trace` dentro del bucle (comprobando la contraseña estimada), estás actualizando el prefijo de la contraseña. Por ejemplo, la versión anterior del código (adivinando un único carácter) tenía este aspecto:

    trace = cap_pass_trace(c + "\n")

¡Pero eso siempre está enviando únicamente nuestro primer carácter! Por lo tanto, necesitamos enviar la "contraseña correcta conocida hasta el momento". En el código de ejemplo, algo como esto:
   
    trace = cap_pass_trace(guessed_pw + c + "\n")

Donde `guessed_pw` crece progresivamente con el inicio correcto ya conocido de la contraseña.

### Siempre se obtiene 'a'

Esto parece indicar que siempre coincide con el primer carácter:

    h
    ha
    haa
    haaa
    haaaa

Comprueba que actualizas `ref_trace`; si reutilizas la traza de referencia original, no estarás comparando contra una referencia donde los primeros N caracteres son correctos y los restantes son incorrectos. Una forma sencilla de hacer esto es, nuevamente, utilizar la variable `guessed_pw` y añadir un byte nulo + salto de línea:

    trace = cap_pass_trace(guessed_pw + "\x00\n")

---
**Agradecimientos / Referencias:**  
Este material está basado y adaptado a partir del notebook original [*Lab 2_1B - Power Analysis for Password Bypass (MAIN).ipynb*](https://github.com/newaetech/chipwhisperer-jupyter/blob/main/courses/sca101/Lab%202_1B%20-%20Power%20Analysis%20for%20Password%20Bypass%20(MAIN).ipynb) del repositorio oficial de [NewAE Technology](https://github.com/newaetech/chipwhisperer-jupyter).